In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [12]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor

# 1. Directory configuration using the path you provided
INPUT_DIR = "/kaggle/input/competitions/stanford-rna-3d-folding-2"

def load_data():
    """Load sequences and labels. Use nrows to manage memory."""
    print("Step 1: Loading data...")
    train_seq = pd.read_csv(f"{INPUT_DIR}/train_sequences.csv")
    test_seq = pd.read_csv(f"{INPUT_DIR}/test_sequences.csv")
    
    try:
        # Loading a sample of the labels
        train_labels = pd.read_csv(f"{INPUT_DIR}/train_labels.csv", nrows=100000)
        
        # IMPORTANT: Drop rows where coordinates are NaN to avoid ValueError
        initial_count = len(train_labels)
        train_labels = train_labels.dropna(subset=['x_1', 'y_1', 'z_1'])
        dropped_count = initial_count - len(train_labels)
        if dropped_count > 0:
            print(f"Dropped {dropped_count} rows containing NaN coordinates.")
            
    except FileNotFoundError:
        print(f"Error: train_labels.csv not found at {INPUT_DIR}")
        train_labels = None
        
    return train_seq, test_seq, train_labels

def extract_features(df):
    """Convert nucleotides (A, C, G, U) to numeric features."""
    mapping = {'A': 0, 'C': 1, 'G': 2, 'U': 3}
    # Map bases; fill any unknown characters with -1
    df['res_encoded'] = df['resname'].map(mapping).fillna(-1)
    return df[['resid', 'res_encoded']]

def main():
    train_seq, test_seq, train_labels = load_data()
    
    # --- Training Phase ---
    model = None
    if train_labels is not None:
        print("Step 2: Training Random Forest model...")
        X_train = extract_features(train_labels)
        y_train = train_labels[['x_1', 'y_1', 'z_1']]
        
        # Initialize and fit the model
        model = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
        model.fit(X_train, y_train)
        print("Model training complete.")

    # --- Prediction Phase ---
    print("Step 3: Generating predictions for test sequences...")
    submission_rows = []
    base_map = {'A': 0, 'C': 1, 'G': 2, 'U': 3}
    
    for _, row in test_seq.iterrows():
        target_id = row['target_id']
        sequence = row['sequence']
        
        for i, nucleotide in enumerate(sequence):
            resid = i + 1
            res_id = f"{target_id}_{resid}"
            
            # Prepare test feature
            res_encoded = base_map.get(nucleotide, -1)
            X_test = np.array([[resid, res_encoded]])
            
            if model:
                # Predict base coordinates
                base_coords = model.predict(X_test)[0]
            else:
                # Fallback: simple spiral geometry
                base_coords = [10 * np.cos(resid * 0.5), 10 * np.sin(resid * 0.5), resid * 2.8]
            
            # Build the submission row for 5 structures
            entry = {'ID': res_id, 'resname': nucleotide, 'resid': resid}
            for s in range(1, 6):
                offset = (s - 1) * 0.2
                entry[f'x_{s}'] = base_coords[0] + offset
                entry[f'y_{s}'] = base_coords[1] + offset
                entry[f'z_{s}'] = base_coords[2] + offset
                
            submission_rows.append(entry)

    # --- Finalizing Submission ---
    print("Step 4: Saving submission.csv...")
    submission_df = pd.DataFrame(submission_rows)
    
    # Clip coordinates (-999.999 to 9999.999) per competition rules
    coord_cols = [c for c in submission_df.columns if c.startswith(('x_', 'y_', 'z_'))]
    submission_df[coord_cols] = submission_df[coord_cols].clip(lower=-999.999, upper=9999.999)
    
    submission_df.to_csv("submission.csv", index=False)
    print("Process complete. submission.csv is created.")
    print(submission_df.head())

if __name__ == "__main__":
    main()

Step 1: Loading data...
Dropped 12972 rows containing NaN coordinates.
Step 2: Training Random Forest model...
Model training complete.
Step 3: Generating predictions for test sequences...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/

Step 4: Saving submission.csv...
Process complete. submission.csv is created.
       ID resname  resid        x_1        y_1        z_1        x_2  \
0  8ZNQ_1       A      1  48.837806  55.786466  48.736289  49.037806   
1  8ZNQ_2       C      2  14.357448  23.377187  26.034381  14.557448   
2  8ZNQ_3       C      3  17.285865  14.424170  15.732498  17.485865   
3  8ZNQ_4       G      4  19.044338  17.627528  24.750754  19.244338   
4  8ZNQ_5       U      5  19.857159  14.392653  18.125766  20.057159   

         y_2        z_2        x_3        y_3        z_3        x_4  \
0  55.986466  48.936289  49.237806  56.186466  49.136289  49.437806   
1  23.577187  26.234381  14.757448  23.777187  26.434381  14.957448   
2  14.624170  15.932498  17.685865  14.824170  16.132498  17.885865   
3  17.827528  24.950754  19.444338  18.027528  25.150754  19.644338   
4  14.592653  18.325766  20.257159  14.792653  18.525766  20.457159   

         y_4        z_4        x_5        y_5        z_5  
0  

Citation
Rhiju Das, Youhan Lee, Chris Munley, Przemek Porebski, Walter Reade, Theo Viel, and Ashley Oldacre. Stanford RNA 3D Folding Part 2. https://kaggle.com/competitions/stanford-rna-3d-folding-2, 2026. Kaggle.